# Week 18: Experiment A, final runs, tables and figures (milestone M7)

You'll follow your `PROTOCOL.md` **exactly**: run every retrieval system on the test split **once**, build the results table with CIs and corrected p-values, make publication-quality figures, and finish the error analysis.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Final configs; run everything from the command line | 2 h |
| Day 2 | `medlab/report.py`: results table with CIs, p-values and Holm correction | 3 h |
| Day 3 | Figures that tell the story | 2.5 h |
| Day 4 | Write up the Experiment A results section | 2 h |
| Review | Spaced recall | 0.5 h |

### Milestone M7
- [ ] `results/retrieval/*.json` for all systems in the protocol, produced by the CLI.
- [ ] `results/table_A.md`, generated by code (never typed by hand).
- [ ] `report/figures/fig_A_*.png` (at least 2 figures).
- [ ] `report/results_A.md` interpreting the table *against your pre-registered hypotheses*.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

---
# Day 1: Final runs (2 h)

### Exercise 18.1
1. Create `configs/bm25_tuned.yaml`, using the `k1`/`b` you selected **on dev** in Week 10 (write the dev nDCG in a YAML comment), and `configs/tfidf.yaml`.
2. Make `medlab/evaluate.py` write to `results/retrieval/<name>.json` (a subfolder, to keep things tidy).
3. Run all five configs: `bm25`, `bm25_tuned`, `tfidf`, `dense`, `hybrid_rrf`. Use the loop below, or write `scripts/run_retrieval.sh`.

From now on, results only ever come from the CLI. **No numbers copied from notebooks.**

In [ ]:
# Create the two configs and update evaluate.py's output folder, then run the next cell.

In [ ]:
import subprocess, sys, json
names = ["bm25", "bm25_tuned", "tfidf", "dense", "hybrid_rrf"]
for n in names:
    out = subprocess.run([sys.executable, "-m", "medlab.evaluate", "--config", f"configs/{n}.yaml"], cwd=CAP, capture_output=True, text=True)
    assert out.returncode == 0, out.stderr[-1500:]
files = sorted((CAP / "results" / "retrieval").glob("*.json"))
print([f.name for f in files])
assert len(files) >= 5
print("✅ 18.1 passed")

---
# Day 2: The results table, generated by code (3 h)

### Exercise 18.2: `capstone/medlab/report.py`
```python
load_results(folder) -> dict[str, dict]          # name (from config) -> loaded JSON
comparison_table(results, baseline, metric="ndcg@10", seed=0) -> pandas.DataFrame
    # one row per system: mean, delta vs baseline, ci_low, ci_high (bootstrap of the paired per-query difference),
    # p (paired randomisation test), significant (Holm over all non-baseline systems). Baseline row: delta 0, the rest empty.
    # Use the queries in the baseline's per_query; sort rows by mean, descending.
to_markdown(df, floatfmt=".3f") -> str           # a simple GitHub-style markdown table, written yourself (no extra dependency)
```
Reuse `medlab.stats`. Write `tests/test_report.py` with a small fake results dict where you know the answer (for example, one system that is identical to the baseline must get p = 1 and delta 0).

In [ ]:
# Implement report.py and tests, then run the checker.

In [ ]:
report = fresh("medlab.report")
assert run_tests("tests/test_report.py") == 0
results = report.load_results(CAP / "results" / "retrieval")
table = report.comparison_table(results, baseline="bm25_default")
print(table)
assert set(["mean", "delta", "ci_low", "ci_high", "p", "significant"]) <= set(table.columns)
assert table.loc["bm25_default", "delta"] == 0
md_table = report.to_markdown(table)
assert md_table.count("|") > 10 and "---" in md_table
(CAP / "results" / "table_A.md").write_text(md_table)
print(md_table)
print("✅ 18.2 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Per-query arrays: `qids = sorted(results[baseline]['per_query'])`, `a = np.array([results[name]['per_query'][q][metric] for q in qids])`.

</details>

<details><summary>Hint 2</summary>

Collect the p-values in a dict, call `holm` once at the end, and map the result back onto the rows.

</details>


---
# Day 3: Figures (2.5 h)

**A good figure has one message.** It should be readable in greyscale, with labelled axes, units, and the n (number of queries) stated somewhere. Make two:
1. **`fig_A_systems.png`**: mean nDCG@10 per system, with **95% bootstrap CIs as error bars** (bootstrap the per-query scores of each system).
2. **`fig_A_delta.png`**: per-query nDCG@10 difference (best system − BM25), sorted, as bars. This shows *how often* and *by how much* each system wins. A mean can hide a system that helps a few queries a lot and hurts many a little.

### Exercise 18.3: `plot_systems(results, metric, path)` and `plot_delta(results, system, baseline, metric, path)` in `report.py`
Use `matplotlib` with `fig.savefig(path, dpi=200, bbox_inches="tight")` and `plt.close(fig)`.

In [ ]:
# Add the two plotting functions, then run the checker.

In [ ]:
report = fresh("medlab.report")
figdir = CAP / "report" / "figures"; figdir.mkdir(parents=True, exist_ok=True)
best = table.drop(index="bm25_default")["mean"].idxmax()
report.plot_systems(results, "ndcg@10", figdir / "fig_A_systems.png")
report.plot_delta(results, best, "bm25_default", "ndcg@10", figdir / "fig_A_delta.png")
from IPython.display import Image, display
for f in ["fig_A_systems.png", "fig_A_delta.png"]:
    assert (figdir / f).stat().st_size > 5000
    display(Image(filename=str(figdir / f), width=500))
print("✅ 18.3 passed")

---
# Day 4: Write the Experiment A results (2 h)

### Exercise 18.4: `capstone/report/results_A.md`
About 400–600 words, with these sections:
- **Setup** (2–3 sentences, pointing to `PROTOCOL.md`),
- **Results**: the table (include `table_A.md`), and for **each hypothesis** say: supported / not supported, with the numbers (Δ, CI, p, Holm),
- **Per-query view**: what the delta figure shows,
- **Error analysis**: a summary of `ERROR_ANALYSIS.md`, updated with the final systems,
- **Deviations from protocol**: *any* change you made after pre-registration, and why. "None" is fine if it's true. Honesty here is a mark of a good researcher.

In [ ]:
# Write report/results_A.md, then run the checker.

In [ ]:
t = (CAP / "report" / "results_A.md").read_text()
for h in ["Setup", "Results", "Per-query", "Error analysis", "Deviations from protocol"]:
    assert h.lower() in t.lower(), f"missing section {h}"
assert len(t.split()) >= 150
print("✅ 18.4 passed (milestone M7)")

---
# Review (30 min)
1. Why bootstrap *per-query* differences rather than comparing two separate CIs?
2. When is a statistically significant improvement *not* worth reporting as a win?
3. *(Week 11)* Why can dense retrieval fail on a nutrition corpus?
4. *(Week 13)* Derive $\partial L / \partial Z_2$ for softmax + cross-entropy again, from memory.
5. *(Week 15)* What's AUROC's weakness for clinical use?

✍️ *Your answer:* (double-click to edit)

